# S6E7 recovery_activity 실험 노트북

목표는 현재 ablation에서 선택된 **원본 13개 + recovery/activity 파생변수 15개**만 사용해 모델과 앙상블 효과를 공정하게 비교하는 것입니다. 외부 데이터와 공개 prediction 파일은 사용하지 않습니다.

실험 순서: EDA·통계검정 → 표본 10모델 비교 → LightGBM 소규모 튜닝 → 전체 데이터 5-fold LGBM/XGBoost/CatBoost → 신경망 다양성 probe → OOF agreement → 산술/기하 평균 → submission 생성.

## 0. 실행 설정

- 처음 구조만 확인하려면 `QUICK_MODE=True`로 실행합니다.
- 실제 비교와 제출 파일은 `QUICK_MODE=False`로 다시 실행합니다.
- Kaggle GPU 사용 시 XGBoost, CatBoost, PyTorch MLP가 GPU를 사용합니다.
- 정확한 RealMLP 전체 구현은 길이가 크므로 이번 1차 실험에서는 동일한 역할의 경량 PyTorch MLP로 모델 다양성만 먼저 검사합니다.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "catboost": "catboost",
    "lightgbm": "lightgbm",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "polars": "polars",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "xgboost": "xgboost",
}
missing_packages = [
    package for module, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    print(f"Installing missing packages into {sys.executable}: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

In [ ]:
from __future__ import annotations

import gc
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Final

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from scipy.stats import chi2_contingency, kruskal
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

SEED: Final = 42
FOLDS = 5
SCREEN_FOLDS: Final = 3
QUICK_MODE = True
RUN_SCREENING = True
RUN_TUNING = True
RUN_MLP = True
SCREEN_ROWS = 30_000 if QUICK_MODE else 120_000
FINAL_ROWS = 80_000 if QUICK_MODE else None
TREE_ESTIMATORS = 100 if QUICK_MODE else 800
MLP_EPOCHS = 1 if QUICK_MODE else 3
USE_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_GPU else "cpu")
RUN_TAG = "quick" if QUICK_MODE else "full"
OUTPUT_ROOT_OVERRIDE: Path | None = None

np.random.seed(SEED)
torch.manual_seed(SEED)
print({"quick_mode": QUICK_MODE, "gpu": USE_GPU, "device": str(DEVICE)})

In [ ]:
TARGET: Final = "health_condition"
ID_COLUMN: Final = "id"
CLASS_ORDER: Final = ("at-risk", "fit", "unhealthy")
CLASS_TO_INDEX: Final = {name: index for index, name in enumerate(CLASS_ORDER)}

DATA_CANDIDATES = (
    Path("/kaggle/input/competitions/playground-series-s6e7"),
    Path("/kaggle/input/playground-series-s6e7"),
    Path.cwd(),
    Path.cwd() / "kaggle_student_classification",
)
DATA_DIR = next(path for path in DATA_CANDIDATES if (path / "train.csv").exists())
DEFAULT_OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else DATA_DIR / "outputs" / "recovery_activity_notebook"
OUTPUT_ROOT = OUTPUT_ROOT_OVERRIDE or DEFAULT_OUTPUT_ROOT
OUTPUT_DIR = OUTPUT_ROOT / RUN_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pl.read_csv(DATA_DIR / "train.csv")
test = pl.read_csv(DATA_DIR / "test.csv")
required = {ID_COLUMN, TARGET}
assert required.issubset(train.columns)
assert ID_COLUMN in test.columns and TARGET not in test.columns
pl.DataFrame({
    "run_tag": [RUN_TAG], "quick_mode": [QUICK_MODE], "folds": [FOLDS],
    "screen_rows": [SCREEN_ROWS], "final_rows": [-1 if FINAL_ROWS is None else FINAL_ROWS],
    "tree_estimators": [TREE_ESTIMATORS], "mlp_epochs": [MLP_EPOCHS],
}).write_csv(OUTPUT_DIR / "run_manifest.csv")
print(f"data_dir={DATA_DIR}")
print(f"train={train.shape}, test={test.shape}")

## 1. 탐색적 자료분석과 통계검정

발표에는 클래스 불균형, 결측률, 주요 수치형 분포와 통계검정 결과를 사용합니다. 표본 검정은 큰 표본에서 사소한 차이도 유의해지는 문제를 줄이기 위해 최대 50,000행으로 제한합니다.

In [ ]:
target_distribution = (
    train.group_by(TARGET).len()
    .with_columns((pl.col("len") / train.height).alias("share"))
    .sort("share", descending=True)
)
missing_summary = pl.DataFrame({
    "feature": train.columns,
    "missing_rate": [train[column].null_count() / train.height for column in train.columns],
}).sort("missing_rate", descending=True)
display(target_distribution)
display(missing_summary.head(15))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(target_distribution[TARGET], target_distribution["share"], color=["#E69F00", "#E24B4A", "#009E73"])
axes[0].set_title("Target distribution")
axes[0].set_ylabel("share")
top_missing = missing_summary.head(13).sort("missing_rate")
axes[1].barh(top_missing["feature"], top_missing["missing_rate"], color="#378ADD")
axes[1].set_title("Missing rate")
plt.tight_layout()
plt.show()

In [ ]:
NUMERIC_COLUMNS: Final = (
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
)
CATEGORICAL_COLUMNS: Final = (
    "diet_type", "stress_level", "sleep_quality",
    "physical_activity_level", "smoking_alcohol", "gender",
)

eda_indices, _ = train_test_split(
    np.arange(train.height),
    train_size=min(50_000, train.height - 1),
    stratify=train[TARGET].to_numpy(),
    random_state=SEED,
)
eda_sample = train[eda_indices]
numeric_tests = []
for column in NUMERIC_COLUMNS:
    groups = [
        eda_sample.filter(pl.col(TARGET) == label)[column].drop_nulls().to_numpy()
        for label in CLASS_ORDER
    ]
    statistic, p_value = kruskal(*groups)
    numeric_tests.append({"feature": column, "test": "Kruskal-Wallis", "statistic": statistic, "p_value": p_value})

categorical_tests = []
for column in CATEGORICAL_COLUMNS:
    table = (
        eda_sample.with_columns(pl.col(column).fill_null("__MISSING__"))
        .group_by([column, TARGET]).len()
        .pivot(on=TARGET, index=column, values="len", aggregate_function="sum")
        .fill_null(0)
    )
    observed = table.select([pl.col(label) if label in table.columns else pl.lit(0).alias(label) for label in CLASS_ORDER]).to_numpy()
    statistic, p_value, _, _ = chi2_contingency(observed)
    categorical_tests.append({"feature": column, "test": "Chi-square", "statistic": statistic, "p_value": p_value})

display(pl.DataFrame(numeric_tests).sort("statistic", descending=True))
display(pl.DataFrame(categorical_tests).sort("statistic", descending=True))

## 2. 선택된 파생변수: recovery + activity

Ablation에서 `0.949600 → 0.949830`으로 개선된 조합만 유지합니다. 생활습관·불일치·비율 변수는 이번 실험 코드에 넣지 않습니다.

In [ ]:
RECOVERY_NUMERIC_COLUMNS: Final = (
    "sleep_deviation_8h", "sleep_deficit", "excess_sleep",
    "sleep_quality_score", "stress_score", "recovery_deficit",
    "poor_recovery_flag", "sleep_stress_interaction",
)
ACTIVITY_NUMERIC_COLUMNS: Final = (
    "bmi_distance_healthy", "activity_volume", "activity_level_score",
)
ACTIVITY_CATEGORICAL_COLUMNS: Final = (
    "bmi_band", "gender_bmi_band", "gender_activity_level", "activity_mode",
)

def add_selected_features(frame: pl.DataFrame) -> pl.DataFrame:
    enriched = frame.with_columns(
        (pl.col("sleep_duration") - 8.0).abs().alias("sleep_deviation_8h"),
        (7.0 - pl.col("sleep_duration")).clip(lower_bound=0.0).alias("sleep_deficit"),
        (pl.col("sleep_duration") - 9.0).clip(lower_bound=0.0).alias("excess_sleep"),
        pl.when(pl.col("sleep_quality") == "poor").then(0.0)
        .when(pl.col("sleep_quality") == "average").then(1.0)
        .when(pl.col("sleep_quality") == "good").then(2.0)
        .otherwise(None).alias("sleep_quality_score"),
        pl.when(pl.col("stress_level") == "low").then(0.0)
        .when(pl.col("stress_level") == "medium").then(1.0)
        .when(pl.col("stress_level") == "high").then(2.0)
        .otherwise(None).alias("stress_score"),
        ((pl.col("sleep_duration") < 7.0) & (pl.col("sleep_quality") == "poor") & (pl.col("stress_level") == "high"))
        .cast(pl.Int8).alias("poor_recovery_flag"),
        pl.when(pl.col("bmi").is_null()).then(None)
        .when(pl.col("bmi") < 18.5).then(18.5 - pl.col("bmi"))
        .when(pl.col("bmi") > 24.9).then(pl.col("bmi") - 24.9)
        .otherwise(0.0).alias("bmi_distance_healthy"),
        (pl.col("step_count") * pl.col("exercise_duration")).alias("activity_volume"),
        pl.when(pl.col("physical_activity_level") == "sedentary").then(0.0)
        .when(pl.col("physical_activity_level") == "moderate").then(1.0)
        .when(pl.col("physical_activity_level") == "active").then(2.0)
        .otherwise(None).alias("activity_level_score"),
        pl.when(pl.col("bmi").is_null()).then(None)
        .when(pl.col("bmi") < 18.5).then(pl.lit("underweight"))
        .when(pl.col("bmi") <= 24.9).then(pl.lit("normal"))
        .when(pl.col("bmi") <= 29.9).then(pl.lit("overweight"))
        .otherwise(pl.lit("high")).alias("bmi_band"),
        pl.when(pl.col("step_count").is_null() | pl.col("exercise_duration").is_null()).then(None)
        .when((pl.col("step_count") >= 10_000) & (pl.col("exercise_duration") >= 45)).then(pl.lit("endurance"))
        .when((pl.col("step_count") < 6_000) & (pl.col("exercise_duration") >= 45)).then(pl.lit("nonwalking"))
        .when(pl.col("step_count") >= 10_000).then(pl.lit("walking"))
        .when(pl.col("exercise_duration") >= 20).then(pl.lit("formal"))
        .otherwise(pl.lit("low")).alias("activity_mode"),
    )
    return enriched.with_columns(
        (pl.col("stress_score") + pl.col("sleep_deficit") + (2.0 - pl.col("sleep_quality_score"))).alias("recovery_deficit"),
        (pl.col("sleep_deficit") * pl.col("stress_score")).alias("sleep_stress_interaction"),
        pl.concat_str(["gender", "bmi_band"], separator="_").alias("gender_bmi_band"),
        pl.concat_str(["gender", "physical_activity_level"], separator="_").alias("gender_activity_level"),
    )

MODEL_NUMERIC_COLUMNS: Final = NUMERIC_COLUMNS + RECOVERY_NUMERIC_COLUMNS + ACTIVITY_NUMERIC_COLUMNS
MODEL_CATEGORICAL_COLUMNS: Final = CATEGORICAL_COLUMNS + ACTIVITY_CATEGORICAL_COLUMNS
FEATURE_COLUMNS: Final = MODEL_NUMERIC_COLUMNS + MODEL_CATEGORICAL_COLUMNS
assert len(FEATURE_COLUMNS) == 28 and len(set(FEATURE_COLUMNS)) == 28

In [ ]:
train_features = add_selected_features(train)
test_features = add_selected_features(test)

def to_object_matrix(frame: pl.DataFrame) -> np.ndarray:
    numeric = frame.select([pl.col(column).cast(pl.Float64) for column in MODEL_NUMERIC_COLUMNS]).to_numpy()
    categorical = frame.select([
        pl.col(column).cast(pl.String).fill_null("__MISSING__")
        for column in MODEL_CATEGORICAL_COLUMNS
    ]).to_numpy()
    return np.concatenate([numeric, categorical], axis=1).astype(object)

all_rows = np.arange(train_features.height)
y_all = train_features[TARGET].replace_strict(CLASS_TO_INDEX).to_numpy().astype(np.int64)
if FINAL_ROWS is None:
    selected_rows = all_rows
else:
    selected_rows, _ = train_test_split(all_rows, train_size=FINAL_ROWS, stratify=y_all, random_state=SEED)

X = to_object_matrix(train_features[selected_rows])
y = y_all[selected_rows]
X_test = to_object_matrix(test_features)
test_ids = test[ID_COLUMN].to_numpy()
NUMERIC_INDEX = list(range(len(MODEL_NUMERIC_COLUMNS)))
CATEGORICAL_INDEX = list(range(len(MODEL_NUMERIC_COLUMNS), len(FEATURE_COLUMNS)))
print(f"model rows={len(y):,}, features={X.shape[1]}, test rows={len(X_test):,}")

## 3. 공통 검증·전처리 Pipeline

모든 최종 후보는 동일한 `StratifiedKFold(5, shuffle=True, random_state=42)`를 사용합니다. 수치형 결측은 트리 최종 모델에서는 유지하고, 범주형 결측만 `__MISSING__` 범주로 보존합니다. 표본 비교와 MLP에서는 모델 요구사항 때문에 fold 내부에서 수치형 중앙값 대체·스케일링을 수행합니다.

In [ ]:
def make_screen_preprocessor() -> Pipeline:
    columns = ColumnTransformer(
        [
            ("numeric", SimpleImputer(strategy="median", add_indicator=True), NUMERIC_INDEX),
            ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL_INDEX),
        ],
        sparse_threshold=0.0,
    )
    return Pipeline([("columns", columns), ("scale", StandardScaler())])

def make_tree_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        [
            ("numeric", "passthrough", NUMERIC_INDEX),
            ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL_INDEX),
        ],
        sparse_threshold=0.0,
    )

def make_mlp_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        [
            ("numeric", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())]), NUMERIC_INDEX),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_INDEX),
        ],
        sparse_threshold=0.0,
    )

def score_probabilities(labels: np.ndarray, probabilities: np.ndarray) -> float:
    return float(balanced_accuracy_score(labels, probabilities.argmax(axis=1)))

def assert_probability_class_order(model: object) -> None:
    actual = np.asarray(model.classes_, dtype=np.int64)
    expected = np.arange(len(CLASS_ORDER), dtype=np.int64)
    if not np.array_equal(actual, expected):
        raise RuntimeError(f"probability class order mismatch: {actual.tolist()}")

full_splitter = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
full_splits = list(full_splitter.split(X, y))
assert len(full_splits) == FOLDS and all(np.unique(y[valid]).size == len(CLASS_ORDER) for _, valid in full_splits)

## 4. 표본 기반 10개 모델 비교

전체 데이터에서 10개 모델을 모두 5-fold로 돌리면 시간이 과도하므로, 가이드라인대로 층화 표본과 3-fold를 사용해 모델별 Balanced Accuracy와 실행시간을 비교합니다. 이 표는 후보 선별용이며 최종 성능표와 분리합니다.

In [ ]:
@dataclass(frozen=True, slots=True)
class ScreenModel:
    name: str
    factory: Callable[[], object]
    uses_sample_weight: bool

SCREEN_MODELS: Final = (
    ScreenModel("LogisticRegression", lambda: LogisticRegression(max_iter=300, class_weight="balanced", random_state=SEED), False),
    ScreenModel("LinearDiscriminantAnalysis", lambda: LinearDiscriminantAnalysis(), False),
    ScreenModel("GaussianNB", lambda: GaussianNB(), True),
    ScreenModel("DecisionTree", lambda: DecisionTreeClassifier(max_depth=14, min_samples_leaf=30, class_weight="balanced", random_state=SEED), False),
    ScreenModel("RandomForest", lambda: RandomForestClassifier(n_estimators=120, min_samples_leaf=10, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED), False),
    ScreenModel("ExtraTrees", lambda: ExtraTreesClassifier(n_estimators=120, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED), False),
    ScreenModel("HistGradientBoosting", lambda: HistGradientBoostingClassifier(max_iter=250, class_weight="balanced", random_state=SEED), False),
    ScreenModel("XGBoost", lambda: XGBClassifier(n_estimators=350, max_depth=8, learning_rate=0.05, subsample=0.9, colsample_bytree=0.8, tree_method="hist", device="cuda" if USE_GPU else "cpu", n_jobs=-1, random_state=SEED), True),
    ScreenModel("LightGBM", lambda: LGBMClassifier(n_estimators=350, learning_rate=0.05, num_leaves=63, min_child_samples=100, n_jobs=-1, verbosity=-1, random_state=SEED), True),
    ScreenModel("CatBoost", lambda: CatBoostClassifier(iterations=350, depth=8, learning_rate=0.05, loss_function="MultiClass", verbose=False, allow_writing_files=False, random_seed=SEED), True),
)
assert len(SCREEN_MODELS) == 10

if SCREEN_ROWS >= len(y):
    screen_rows = np.arange(len(y))
else:
    screen_rows, _ = train_test_split(
        np.arange(len(y)), train_size=SCREEN_ROWS, stratify=y, random_state=SEED,
    )
X_screen, y_screen = X[screen_rows], y[screen_rows]
screen_splits = list(StratifiedKFold(n_splits=SCREEN_FOLDS, shuffle=True, random_state=SEED).split(X_screen, y_screen))
screen_records = []

if RUN_SCREENING:
    for specification in SCREEN_MODELS:
        started = time.perf_counter()
        scores = []
        for training_rows, validation_rows in screen_splits:
            preprocessor = make_screen_preprocessor()
            X_training = preprocessor.fit_transform(X_screen[training_rows])
            X_validation = preprocessor.transform(X_screen[validation_rows])
            model = specification.factory()
            if specification.uses_sample_weight:
                model.fit(X_training, y_screen[training_rows], sample_weight=compute_sample_weight("balanced", y_screen[training_rows]))
            else:
                model.fit(X_training, y_screen[training_rows])
            scores.append(float(balanced_accuracy_score(y_screen[validation_rows], model.predict(X_validation))))
        screen_records.append({
            "model": specification.name,
            "balanced_accuracy_mean": float(np.mean(scores)),
            "balanced_accuracy_std": float(np.std(scores)),
            "elapsed_seconds": time.perf_counter() - started,
        })
        print(screen_records[-1])

screen_results = pl.DataFrame(screen_records).sort("balanced_accuracy_mean", descending=True) if screen_records else pl.DataFrame()
if screen_records:
    screen_results.write_csv(OUTPUT_DIR / "screening_10_models.csv")
    display(screen_results)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ordered = screen_results.sort("balanced_accuracy_mean")
    axes[0].barh(ordered["model"], ordered["balanced_accuracy_mean"], color="#378ADD")
    axes[0].set_title("Sample CV balanced accuracy")
    axes[1].barh(ordered["model"], ordered["elapsed_seconds"], color="#E69F00")
    axes[1].set_title("Elapsed seconds")
    plt.tight_layout()
    plt.show()

## 5. LightGBM 소규모 하이퍼파라미터 튜닝

리더보드가 아니라 동일한 층화 표본 CV에서 `num_leaves`와 `min_child_samples` 네 조합만 비교합니다. 이 결과는 튜닝 방법 설명용 보조 결과이며, 최종 OOF에는 사전 고정한 파라미터를 사용해 같은 표본에서의 선택 편향을 막습니다.

In [ ]:
LGB_CANDIDATES: Final = (
    {"num_leaves": 31, "min_child_samples": 100},
    {"num_leaves": 63, "min_child_samples": 100},
    {"num_leaves": 63, "min_child_samples": 200},
    {"num_leaves": 127, "min_child_samples": 200},
)
tuning_records = []
if RUN_TUNING:
    for parameters in LGB_CANDIDATES:
        scores = []
        started = time.perf_counter()
        for training_rows, validation_rows in screen_splits:
            preprocessor = make_tree_preprocessor()
            X_training = preprocessor.fit_transform(X_screen[training_rows])
            X_validation = preprocessor.transform(X_screen[validation_rows])
            training_weight = compute_sample_weight("balanced", y_screen[training_rows])
            validation_weight = compute_sample_weight("balanced", y_screen[validation_rows])
            model = LGBMClassifier(
                objective="multiclass", num_class=3, n_estimators=150 if QUICK_MODE else 800,
                learning_rate=0.05, feature_fraction=0.8, bagging_fraction=0.9, bagging_freq=1,
                n_jobs=-1, verbosity=-1, random_state=SEED, **parameters,
            )
            model.fit(
                X_training, y_screen[training_rows], sample_weight=training_weight,
                eval_set=[(X_validation, y_screen[validation_rows])],
                eval_sample_weight=[validation_weight],
                callbacks=[lgb.early_stopping(50, verbose=False)],
            )
            scores.append(score_probabilities(y_screen[validation_rows], model.predict_proba(X_validation)))
        tuning_records.append({**parameters, "balanced_accuracy": float(np.mean(scores)), "elapsed_seconds": time.perf_counter() - started})

tuning_results = pl.DataFrame(tuning_records).sort("balanced_accuracy", descending=True) if tuning_records else pl.DataFrame()
if tuning_records:
    tuning_results.write_csv(OUTPUT_DIR / "lgb_tuning.csv")
    display(tuning_results)
    best_tuning = tuning_results.row(0, named=True)
    TUNED_LGB_PARAMS = {"num_leaves": int(best_tuning["num_leaves"]), "min_child_samples": int(best_tuning["min_child_samples"])}
else:
    TUNED_LGB_PARAMS = {"num_leaves": 63, "min_child_samples": 100}
FINAL_LGB_PARAMS = {"num_leaves": 63, "min_child_samples": 100}
print("sample-only tuned params", TUNED_LGB_PARAMS)
print("fixed final-CV params", FINAL_LGB_PARAMS)

## 6. 전체 데이터 5-fold: LGBM, XGBoost, CatBoost

세 모델 모두 같은 fold와 사전에 고정한 반복 횟수를 사용합니다. 외부 validation fold를 조기종료나 checkpoint 선택에 다시 사용하지 않습니다. LGBM/XGBoost는 fold 내부 OrdinalEncoder를, CatBoost는 원래 범주형 문자열과 `__MISSING__` 범주를 사용합니다. OOF와 test 확률을 저장해 이후 앙상블을 재학습 없이 반복할 수 있게 합니다.

In [ ]:
oof_probabilities: dict[str, np.ndarray] = {}
test_probabilities: dict[str, np.ndarray] = {}
fold_scores: dict[str, list[float]] = {}
elapsed_seconds: dict[str, float] = {}

def run_encoded_tree_cv(model_name: str) -> None:
    started = time.perf_counter()
    oof = np.zeros((len(y), len(CLASS_ORDER)), dtype=np.float64)
    test_prediction = np.zeros((len(X_test), len(CLASS_ORDER)), dtype=np.float64)
    scores = []
    for fold, (training_rows, validation_rows) in enumerate(full_splits, start=1):
        preprocessor = make_tree_preprocessor()
        X_training = preprocessor.fit_transform(X[training_rows])
        X_validation = preprocessor.transform(X[validation_rows])
        X_test_fold = preprocessor.transform(X_test)
        training_weight = compute_sample_weight("balanced", y[training_rows])
        match model_name:
            case "lgbm":
                model = LGBMClassifier(
                    objective="multiclass", num_class=3, n_estimators=TREE_ESTIMATORS, learning_rate=0.03,
                    feature_fraction=0.8, bagging_fraction=0.9, bagging_freq=1,
                    n_jobs=-1, verbosity=-1, random_state=SEED + fold, **FINAL_LGB_PARAMS,
                )
                model.fit(X_training, y[training_rows], sample_weight=training_weight)
            case "xgboost":
                model = XGBClassifier(
                    objective="multi:softprob", num_class=3, n_estimators=TREE_ESTIMATORS,
                    learning_rate=0.04, max_depth=8, min_child_weight=8, subsample=0.9, colsample_bytree=0.8,
                    reg_alpha=0.1, reg_lambda=1.0, tree_method="hist", device="cuda" if USE_GPU else "cpu",
                    n_jobs=-1, random_state=SEED + fold,
                )
                model.fit(X_training, y[training_rows], sample_weight=training_weight, verbose=False)
            case _:
                raise KeyError(model_name)
        assert_probability_class_order(model)
        oof[validation_rows] = model.predict_proba(X_validation)
        test_prediction += model.predict_proba(X_test_fold) / FOLDS
        score = score_probabilities(y[validation_rows], oof[validation_rows])
        scores.append(score)
        print(f"{model_name} fold={fold} BA={score:.6f}")
        del X_training, X_validation, X_test_fold, model
        gc.collect()
    oof_probabilities[model_name] = oof
    test_probabilities[model_name] = test_prediction
    fold_scores[model_name] = scores
    elapsed_seconds[model_name] = time.perf_counter() - started

run_encoded_tree_cv("lgbm")
run_encoded_tree_cv("xgboost")

In [ ]:
started = time.perf_counter()
cat_oof = np.zeros((len(y), len(CLASS_ORDER)), dtype=np.float64)
cat_test = np.zeros((len(X_test), len(CLASS_ORDER)), dtype=np.float64)
cat_scores = []
for fold, (training_rows, validation_rows) in enumerate(full_splits, start=1):
    model = CatBoostClassifier(
        iterations=TREE_ESTIMATORS, depth=8, learning_rate=0.05, loss_function="MultiClass",
        eval_metric="MultiClass", auto_class_weights="Balanced",
        task_type="GPU" if USE_GPU else "CPU", random_seed=SEED + fold,
        verbose=False, allow_writing_files=False,
    )
    model.fit(X[training_rows], y[training_rows], cat_features=CATEGORICAL_INDEX)
    assert_probability_class_order(model)
    cat_oof[validation_rows] = model.predict_proba(X[validation_rows])
    cat_test += model.predict_proba(X_test) / FOLDS
    score = score_probabilities(y[validation_rows], cat_oof[validation_rows])
    cat_scores.append(score)
    print(f"catboost fold={fold} BA={score:.6f}")
    del model
    gc.collect()

oof_probabilities["catboost"] = cat_oof
test_probabilities["catboost"] = cat_test
fold_scores["catboost"] = cat_scores
elapsed_seconds["catboost"] = time.perf_counter() - started

## 7. 신경망 다양성 probe

첨부 RealMLP의 핵심 목적은 GBDT와 다른 오류 패턴을 얻는 것입니다. 이번 노트북에서는 fold 내부 one-hot/표준화와 class-weighted loss를 사용하는 작은 PyTorch MLP로 그 가능성을 먼저 확인합니다. 단독 점수보다 tree 모델과의 agreement가 낮고 blend가 개선되는지가 채택 기준입니다.

In [ ]:
def predict_mlp(model: nn.Module, features: np.ndarray) -> np.ndarray:
    model.eval()
    batches = []
    with torch.no_grad():
        for start in range(0, len(features), 8_192):
            batch = torch.from_numpy(features[start:start + 8_192]).to(DEVICE)
            batches.append(torch.softmax(model(batch), dim=1).cpu().numpy())
    return np.concatenate(batches)

if RUN_MLP:
    started = time.perf_counter()
    mlp_oof = np.zeros((len(y), len(CLASS_ORDER)), dtype=np.float64)
    mlp_test = np.zeros((len(X_test), len(CLASS_ORDER)), dtype=np.float64)
    mlp_scores = []
    for fold, (training_rows, validation_rows) in enumerate(full_splits, start=1):
        torch.manual_seed(SEED + fold)
        preprocessor = make_mlp_preprocessor()
        X_training = preprocessor.fit_transform(X[training_rows]).astype(np.float32)
        X_validation = preprocessor.transform(X[validation_rows]).astype(np.float32)
        X_test_fold = preprocessor.transform(X_test).astype(np.float32)
        train_dataset = TensorDataset(torch.from_numpy(X_training), torch.from_numpy(y[training_rows]))
        train_loader = DataLoader(train_dataset, batch_size=2_048, shuffle=True, pin_memory=USE_GPU)
        model = nn.Sequential(
            nn.Linear(X_training.shape[1], 256), nn.BatchNorm1d(256), nn.SiLU(), nn.Dropout(0.08),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.SiLU(), nn.Dropout(0.05),
            nn.Linear(256, len(CLASS_ORDER)),
        ).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
        class_weights = compute_class_weight("balanced", classes=np.arange(len(CLASS_ORDER)), y=y[training_rows])
        criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
        for epoch in range(MLP_EPOCHS):
            model.train()
            for feature_batch, label_batch in train_loader:
                feature_batch = feature_batch.to(DEVICE, non_blocking=True)
                label_batch = label_batch.to(DEVICE, non_blocking=True)
                optimizer.zero_grad()
                loss = criterion(model(feature_batch), label_batch)
                loss.backward()
                optimizer.step()
            validation_probability = predict_mlp(model, X_validation)
            epoch_score = score_probabilities(y[validation_rows], validation_probability)
            print(f"torch_mlp fold={fold} epoch={epoch + 1} BA={epoch_score:.6f}")
        mlp_oof[validation_rows] = predict_mlp(model, X_validation)
        mlp_test += predict_mlp(model, X_test_fold) / FOLDS
        mlp_scores.append(score_probabilities(y[validation_rows], mlp_oof[validation_rows]))
        del X_training, X_validation, X_test_fold, train_dataset, train_loader, model
        gc.collect()
        if USE_GPU:
            torch.cuda.empty_cache()
    oof_probabilities["torch_mlp"] = mlp_oof
    test_probabilities["torch_mlp"] = mlp_test
    fold_scores["torch_mlp"] = mlp_scores
    elapsed_seconds["torch_mlp"] = time.perf_counter() - started

## 8. OOF 비교, disagreement, 산술·기하 앙상블

가중치를 Public LB로 정하지 않고, 동일한 OOF 모델들의 **고정 동일 가중치** 산술평균과 geometric probability 평균만 비교합니다. 모델 간 argmax agreement가 낮으면서 blend 성능이 오르는 조합이 실제로 가치 있는 다양성입니다.

In [ ]:
def arithmetic_blend(probabilities: list[np.ndarray]) -> np.ndarray:
    return np.mean(np.stack(probabilities), axis=0)

def geometric_blend(probabilities: list[np.ndarray]) -> np.ndarray:
    log_average = np.mean(np.log(np.clip(np.stack(probabilities), 1e-7, 1.0)), axis=0)
    blended = np.exp(log_average)
    return blended / blended.sum(axis=1, keepdims=True)

model_names = list(oof_probabilities)
oof_list = [oof_probabilities[name] for name in model_names]
test_list = [test_probabilities[name] for name in model_names]
oof_probabilities["arithmetic_blend"] = arithmetic_blend(oof_list)
test_probabilities["arithmetic_blend"] = arithmetic_blend(test_list)
oof_probabilities["geometric_blend"] = geometric_blend(oof_list)
test_probabilities["geometric_blend"] = geometric_blend(test_list)

agreement = np.array([
    [(oof_probabilities[left].argmax(1) == oof_probabilities[right].argmax(1)).mean() for right in model_names]
    for left in model_names
])
agreement_frame = pl.DataFrame(agreement, schema=model_names).insert_column(0, pl.Series("model", model_names))
agreement_frame.write_csv(OUTPUT_DIR / "agreement.csv")
display(agreement_frame)

summary_records = []
for name, probabilities in oof_probabilities.items():
    predictions = probabilities.argmax(1)
    recalls = recall_score(y, predictions, labels=np.arange(len(CLASS_ORDER)), average=None)
    summary_records.append({
        "model": name,
        "oof_balanced_accuracy": score_probabilities(y, probabilities),
        "fold_mean": float(np.mean(fold_scores[name])) if name in fold_scores else None,
        "fold_std": float(np.std(fold_scores[name])) if name in fold_scores else None,
        "recall_at_risk": float(recalls[0]),
        "recall_fit": float(recalls[1]),
        "recall_unhealthy": float(recalls[2]),
        "elapsed_seconds": elapsed_seconds.get(name),
    })
summary = pl.DataFrame(summary_records).sort("oof_balanced_accuracy", descending=True)
summary.write_csv(OUTPUT_DIR / "oof_summary.csv")
display(summary)

best_name = summary["model"][0]
best_probability = test_probabilities[best_name]
submission = pl.DataFrame({
    ID_COLUMN: test_ids,
    TARGET: [CLASS_ORDER[index] for index in best_probability.argmax(1)],
})
submission.write_csv(OUTPUT_DIR / "submission.csv")
for name in model_names:
    np.save(OUTPUT_DIR / f"oof_{name}.npy", oof_probabilities[name])
    np.save(OUTPUT_DIR / f"test_{name}.npy", test_probabilities[name])
print(f"selected={best_name}, submission={OUTPUT_DIR / 'submission.csv'}")
display(submission.group_by(TARGET).len().sort("len", descending=True))

## 9. 발표자료에 옮길 내용

### SCQA
- **Situation:** 생활·수면·활동 데이터로 세 건강상태를 예측한다.
- **Complication:** `at-risk`가 약 86%인 불균형 다중분류이며 결측이 존재한다.
- **Question:** 임상적 의미가 있는 회복·활동 파생변수와 모델 다양성이 Balanced Accuracy를 개선하는가?
- **Answer:** baseline 대비 파생변수 ablation, 표본 10모델 비교, 동일 5-fold OOF와 앙상블 결과로 답한다.

### 필수 표·그림
1. target 및 결측률 시각화
2. Kruskal-Wallis/Chi-square 통계검정표
3. 10개 모델의 점수·시간 비교표
4. 최종 모델별 5-fold OOF, 클래스별 recall, 시간표
5. 모델 agreement 행렬과 산술/기하 앙상블 비교
6. 최종 submission의 ID·예측분포 및 Kaggle 제출 점수 캡처

### 한계와 다음 액션
- 표본 screening 순위는 전체 데이터 순위와 다를 수 있다.
- GPU CatBoost는 비결정적일 수 있어 주요 결과는 seed 반복 검증이 필요하다.
- 이번 MLP가 blend를 개선할 때만 정확한 RealMLP를 같은 5-fold로 재실행한다.
- 최고 후보 선택 자체가 같은 OOF를 사용하므로 최종 확정 전 다른 seed 또는 별도 meta 검증을 수행한다.
- Public prediction 앙상블은 사용하지 않는다. 모든 비교와 제출은 이 노트북에서 학습한 모델만 사용한다.

### 프로그램화 Action Plan
- 사용자 또는 웨어러블 CSV의 **raw 13개 입력**을 받는다.
- 학습 때 사용한 **동일한 파생변수 함수**와 결측 처리로 28개 입력을 생성한다.
- 확정된 전처리기와 모델을 하나의 **모델 artifact**로 저장하고 버전을 기록한다.
- 배치 프로그램에서 먼저 재현한 뒤, 필요하면 **API 또는 UI**로 건강상태와 주의 문구를 제공한다. 의료 진단이 아닌 위험도 참고 도구로 사용 범위를 제한한다.